In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import os

# Visual Style
plt.rcdefaults()
plt.style.use('tableau-colorblind10')
%matplotlib widget 
plt.rcParams.update({'font.size': 10, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black'})

In [ ]:
# --- CONFIGURAÇÃO ---
# Caminho para o snapshot gerado pelo seu script
MINI_SIM_PATH = "./mini_fiducial_test"
SNAP_NUM = "264"
FILE_PATH = os.path.join(MINI_SIM_PATH, f"snapdir_{SNAP_NUM}", f"snapshot_{SNAP_NUM}.0.hdf5")

print(f"--> Lendo arquivo: {FILE_PATH}")

In [ ]:
# --- 1. CARREGAR DADOS ---
with h5py.File(FILE_PATH, 'r') as f:
    # Ler Header
    header = f['Header'].attrs
    box_size = header['BoxSize']
    redshift = header['Redshift']
    time = header['Time']
    n_part = header['NumPart_Total'][0] # Gás é o índice 0
    
    print("-" * 30)
    print(f"INFO DO HEADER:")
    print(f"BoxSize: {box_size:.4f} cMpc/h")
    print(f"Redshift: {redshift:.4f}")
    print(f"Num Partículas de Gás: {n_part}")
    print("-" * 30)
    
    # Ler Gás (PartType0)
    # Coordinates e Density são essenciais para o mapa
    if 'PartType0' in f:
        coords = f['PartType0']['Coordinates'][:]
        # Tenta ler densidade para ponderar o mapa (fica mais bonito/físico)
        if 'Density' in f['PartType0']:
            density = f['PartType0']['Density'][:]
        else:
            print("Aviso: Campo 'Density' não encontrado. Usando contagem de partículas.")
            density = np.ones(len(coords))
    else:
        raise ValueError("ERRO: PartType0 (Gás) não encontrado no arquivo!")

In [ ]:
# --- 2. VERIFICAÇÃO GEOMÉTRICA ---
center = box_size / 2.0
print(f"Centro Esperado da Caixa: {center:.4f}")
print(f"Mínima Coord X: {coords[:,0].min():.4f}")
print(f"Máxima Coord X: {coords[:,0].max():.4f}")

# Checagem rápida se tem algo no centro
dist_from_center = np.sqrt(np.sum((coords - center)**2, axis=1))
print(f"Partícula mais próxima do centro está a: {dist_from_center.min():.4f} cMpc/h")

In [ ]:
# --- 3. PLOTAGEM (MAPA DE GÁS) ---
fig, ax = plt.subplots(figsize=(10, 8))

# Histograma 2D
# Eixo X e Y (Projeção em Z)
# Weights=density faz com que a cor seja proporcional à densidade do gás, não apenas contagem
# Bins=500 garante boa resolução
im = ax.hist2d(
    coords[:, 0], 
    coords[:, 1], 
    bins=500, 
    range=[[0, box_size], [0, box_size]],
    norm=LogNorm(), 
    weights=density,
    cmap='inferno'
)

# Adicionar barra de cores
cbar = plt.colorbar(im[3], ax=ax)
cbar.set_label(r'Densidade Projetada (unidades de código)')

# --- ELEMENTOS VISUAIS ---
# Desenhar uma cruz no centro exato da caixa
ax.scatter(center, center, color='cyan', s=100, marker='+', label='Centro da Nova Caixa')

# Desenhar um círculo representando o R200 original (apenas estimativa visual)
# Sabemos que o recorte foi 10x R200. Então o R200 é 1/10 do raio da caixa (1/20 do boxsize)
r200_est = (box_size / 2) / 10.0 # Aproximação baseada no seu script (R200_FACTOR=10)
circle = plt.Circle((center, center), r200_est, color='white', fill=False, linestyle='--', label='~R200 Original')
ax.add_patch(circle)

# Configurações do gráfico
ax.set_facecolor('black')
ax.set_aspect('equal')
ax.set_xlabel('x [cMpc/h]')
ax.set_ylabel('y [cMpc/h]')
ax.set_title(f"Mini-Simulação: Halo Centralizado (z={redshift:.2f})")
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import os

# Visual Style
plt.rcdefaults()
plt.style.use('tableau-colorblind10')
# %matplotlib widget  # Uncomment if in Jupyter Lab/Notebook
plt.rcParams.update({'font.size': 10, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black'})

# --- CONFIGURATION ---
MINI_SIM_ROOT = "./mini_sims_library_TNG50-4" # The root library folder
SNAP_NUM = "025"
HALO_ID = 1000  # <--- CRITICAL: You must choose which halo to visualize (0, 1, 2...)

# Construct the correct path: ROOT / halo_ID / snapdir / snap
FILE_PATH = os.path.join(MINI_SIM_ROOT, f"halo_{HALO_ID}", f"snapdir_{SNAP_NUM}", f"snapshot_{SNAP_NUM}.0.hdf5")

print(f"--> Reading file: {FILE_PATH}")


# --- 1. LOAD DATA ---
with h5py.File(FILE_PATH, 'r') as f:
    # Read Header
    header = f['Header'].attrs
    raw_box_size = header['BoxSize']
    redshift = header['Redshift']
    time_a = header['Time']
    
    # --- UNIT DETECTION & CONVERSION ---
    # TNG is usually in ckpc/h (Box > 2000). MN5 in cMpc/h.
    # We convert everything to cMpc/h for plotting clarity.
    if raw_box_size > 1000:
        print("--> Detected Native Units: ckpc/h. Converting to cMpc/h for plot.")
        scale_factor = 1.0 / 1000.0
        unit_label = "cMpc/h"
    else:
        print("--> Detected Native Units: cMpc/h.")
        scale_factor = 1.0
        unit_label = "cMpc/h"

    box_size_mpc = raw_box_size * scale_factor

    print("-" * 30)
    print(f"HEADER INFO:")
    print(f"Native Box: {raw_box_size:.1f}")
    print(f"Plot Box  : {box_size_mpc:.2f} {unit_label}")
    print(f"Redshift  : {redshift:.4f}")
    print("-" * 30)
    
    # Read Gas (PartType0)
    if 'PartType0' in f:
        # Load and immediately convert coordinates to Mpc
        coords_raw = f['PartType0']['Coordinates'][:]
        coords_mpc = coords_raw * scale_factor
        
        # Load Density (if available) for better visualization
        if 'Density' in f['PartType0']:
            density = f['PartType0']['Density'][:]
        else:
            print("Warning: 'Density' field not found. Using particle count.")
            density = None # hist2d handles None by just counting
    else:
        raise ValueError("ERROR: PartType0 (Gas) not found in file!")

In [ ]:
# --- 2. GEOMETRY CHECK ---
center_mpc = box_size_mpc / 2.0
print(f"Expected Center: {center_mpc:.4f} {unit_label}")
print(f"Min X: {coords_mpc[:,0].min():.4f}")
print(f"Max X: {coords_mpc[:,0].max():.4f}")

# --- 3. PLOTTING ---
fig, ax = plt.subplots(figsize=(8, 7))

# 2D Histogram (Projected Gas Density)
# Using weights=density makes it a physical density map projection
im = ax.hist2d(
    coords_mpc[:, 0], 
    coords_mpc[:, 1], 
    bins=400, 
    range=[[0, box_size_mpc], [0, box_size_mpc]],
    norm=LogNorm(), 
    weights=density,
    cmap='inferno'
)

# Colorbar
cbar = plt.colorbar(im[3], ax=ax)
cbar.set_label(f'Projected Density (Code Units)')

# Visual Elements
# Center Cross
ax.scatter(center_mpc, center_mpc, color='cyan', s=100, marker='+', label='Box Center')

# Visual R200 estimate (Just for reference)
# Assuming the box was cut as 10 * R200 (FIXED_BOX_MPC logic might differ, but this is visual)
# If using FIXED_BOX_MPC = 150, the box is 150 Mpc.
# The halo is usually small in the center.
circle = plt.Circle((center_mpc, center_mpc), 0.5, color='white', fill=False, linestyle='--', label='~1 Mpc (Visual)')
ax.add_patch(circle)

# Styling
ax.set_facecolor('black')
ax.set_aspect('equal')
ax.set_xlabel(f'x [{unit_label}]')
ax.set_ylabel(f'y [{unit_label}]')
ax.set_title(f"Mini-Sim Halo {HALO_ID} (z={redshift:.2f})\nFull Box: {box_size_mpc:.1f} {unit_label}")
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()